# 교사 라벨링 (Colab GPU)

교사 Qwen3.8-27B 를 API 대신 **Colab GPU 에 직접 올려** 태깅·추출 정답을 만든다. `scripts/label_with_teacher.py` 를 그대로 쓰고,
연결 주소만 OpenRouter → 이 노트북의 vLLM 서버로 바꾼다.

| 순서 | 셀 | 하는 일 |
|---|---|---|
| 1 | GPU 확인 | **A100(40GB) 이상 필요.** L4·T4 는 모델(FP8 약 31GB)이 안 들어간다 |
| 2 | ⚙️ 설정 | 라벨링할 세션 파일, 결과 저장 위치 |
| 3 | 준비 | 드라이브 연결, 저장소 클론, vLLM · 프로젝트 의존성 설치 |
| 4 | 교사 서버 켜기 | vLLM 으로 `Qwen/Qwen3.8-27B-FP8` 서빙 (처음엔 다운로드 포함 10~20분) |
| 5 | **골드셋 점검** | 골드셋 63건을 라벨링해 API 로 만든 라벨과 점수를 비교 — **통과해야 6번으로** |
| 6 | 라벨링 | 합성 세션 전체. 끊기면 다시 실행하면 이어서 한다 |
| 7 | 학습 데이터로 변환 | 태깅 만장일치 · 추출 → `sft/` (학습 노트북 입력) |

**API 와 다른 점:** 같은 모델이지만 **FP8 양자화**본이라 라벨이 아주 조금 다를 수 있다. 그래서 5번에서 골드셋 점수를 먼저 확인한다.
추론(생각) 모드는 API 때처럼 끈다 (`label_with_teacher.py` 가 `enable_thinking: false` 를 보낸다).

**비용:** Colab 컴퓨팅 단위만 든다. 6번 셀이 끝나면 걸린 시간을 보고 1,000세션 기준 시간을 계산해 준다.

## 1. GPU 확인

In [ ]:
import subprocess

import torch

print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True
    ).stdout
)
total_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
assert total_gb >= 39, f"GPU 메모리 {total_gb:.0f}GB — A100(40GB) 이상으로 바꾸세요 (런타임 → 런타임 유형 변경)"

## 2. ⚙️ 설정

| 값 | 뜻 |
|---|---|
| `SESSIONS` | 라벨링할 합성 세션 JSONL (`generate_synthetic_onboarding.py` 또는 `manual_simulator.py import` 출력) |
| `OUT_DIR` | 라벨 결과 폴더. 드라이브에 둬야 세션이 끊겨도 남는다 |
| `CONCURRENCY` | 동시 요청 수. 높을수록 빠르다 (vLLM 이 한꺼번에 묶어 처리) |

In [ ]:
BRANCH = "feat/lena-lora-qlora-compare"
SESSIONS = "/content/drive/MyDrive/ktb-ai/synthetic/manual-run300.jsonl"
OUT_DIR = "/content/drive/MyDrive/ktb-ai/teacher/manual-run300"
GOLD_OUT_DIR = "/content/drive/MyDrive/ktb-ai/teacher/gold-qwen-colab"
SFT_DIR = "/content/drive/MyDrive/ktb-ai/sft"  # 7번 결과 — 학습 노트북의 SFT_ROOT
MODEL = "Qwen/Qwen3.8-27B-FP8"
REVISION = "017b9c7af6b5689d5dd426a76e0bc077eb5ca20a"
CONCURRENCY = 32
API_KEY = "local-teacher"  # 이 노트북 안에서만 쓰는 서버 키

## 3. 준비 — 드라이브, 저장소, 설치

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
print(("✅ " if os.path.exists(SESSIONS) else "❌ 없음: ") + SESSIONS)

!rm -rf /content/repo && git clone -q --depth 1 -b {BRANCH} https://github.com/100-hours-a-week/KTB4-17th-AI /content/repo
%cd /content/repo
!pip install -q vllm==0.30.0
!pip install -q uv && uv sync --frozen -q

## 4. 교사 서버 켜기
처음 실행하면 모델(약 31GB)을 내려받느라 오래 걸린다. 아래 셀이 `서버 준비됨` 을 찍을 때까지 기다린다.

In [ ]:
import time
import urllib.request

server = subprocess.Popen(
    [
        "vllm",
        "serve",
        MODEL,
        "--revision",
        REVISION,
        "--served-model-name",
        "teacher",
        "--max-model-len",
        "8192",
        "--enable-prefix-caching",
        "--gpu-memory-utilization",
        "0.92",
        "--limit-mm-per-prompt",
        '{"image": 0, "video": 0}',
        "--api-key",
        API_KEY,
        "--port",
        "8000",
    ],
    stdout=open("/content/vllm.log", "w"),
    stderr=subprocess.STDOUT,
)
for _ in range(240):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=5)
        print("서버 준비됨")
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("서버가 꺼졌습니다. /content/vllm.log 끝부분을 확인하세요")
        time.sleep(10)
else:
    raise TimeoutError("40분 안에 안 떴습니다. /content/vllm.log 를 확인하세요")

os.environ |= {
    "TEACHER_LLM_BASE_URL": "http://localhost:8000/v1",
    "TEACHER_LLM_MODEL": "teacher",
    "TEACHER_LLM_API_KEY": API_KEY,
    "LANGFUSE_TRACING_ENABLED": "false",
}

## 5. 골드셋 점검 — 통과해야 6번으로
골드셋(calibration + regression) 63건을 같은 방식(5회 샘플)으로 라벨링하고 API 라벨과 비교한다.

| 기준 (API, OpenRouter BF16) | primary F1 | 완전 일치 | 추출 차원 판정 |
|---|---|---|---|
| `gold-qwen-v2` | 83.1 | 71.7 | 100 |

**판정:** 위 값에서 몇 건(F1 약 3점) 안쪽이면 통과. 크게 낮으면 6번으로 가지 말고 결과를 공유한다.

In [ ]:
!uv run python scripts/label_with_teacher.py --gold --out-dir {GOLD_OUT_DIR} --concurrency {CONCURRENCY}
!uv run python scripts/score_teacher_gold.py {GOLD_OUT_DIR} --out /content/gold_score.json > /dev/null

import json

r = json.load(open("/content/gold_score.json"))["results"][0]
t, e = r["tagging"], r["extraction"]
print(f"태깅 primary F1 {100 * t['primaryF1']:.1f} (API 83.1) · 완전 일치 {100 * t['exactMatch']:.1f} (API 71.7)")
print(f"추출 차원 판정 {100 * e['dimensionDecisionAccuracy']:.1f} (API 100) · 범위 안 {e['inRange']} (API 24/24)")

## 6. 라벨링
`--resume` 이라 세션이 끊기거나 멈추면 이 셀을 다시 실행하면 된다(이미 된 건 건너뜀). 끝나면 `OUT_DIR/summary.json` 이 생긴다.

In [ ]:
!uv run python scripts/label_with_teacher.py --synthetic {SESSIONS} --out-dir {OUT_DIR} --concurrency {CONCURRENCY} --resume > /content/label.log

s = json.load(open(f"{OUT_DIR}/summary.json"))
sessions = s["labeledNow"].get("extraction", 0)
print(json.dumps(s["quality"], ensure_ascii=False, indent=1))
if sessions:
    per_1000 = s["elapsedSeconds"] / sessions * 1000 / 3600
    print(f"이번 {sessions}세션 {s['elapsedSeconds'] / 60:.0f}분 → 1,000세션이면 약 {per_1000:.1f}시간")

## 7. 학습 데이터로 변환
태깅은 **만장일치(5/5)**, 추출은 허용 차원 밖 키를 지워 `SFT_DIR/{tagging,extraction}/out/` 에 쓴다. 학습 노트북은 이 폴더를 그대로 읽는다.
시드 충실도 점검(`check_synthetic_labels.py`)도 같이 돌린다 — `bucketAgreement` 0.8 이상이면 시뮬레이터 통과.

In [ ]:
!uv run python scripts/check_synthetic_labels.py {OUT_DIR} --sessions {SESSIONS} | head -40
!uv run python training/tagging/from_teacher.py {OUT_DIR}/tagging.jsonl --out /content/tagging.jsonl --min-vote-share 1.0
!uv run python training/tagging/build_sft.py /content/tagging.jsonl --out {SFT_DIR}/tagging/out --allow-unreviewed
!uv run python training/extraction/from_teacher.py {OUT_DIR}/extraction.jsonl --out {SFT_DIR}/extraction/out
server.terminate()